### Setup the entire notebook's installation and logger

#### First of all, make sure to adjust the BASE_FOLDER_LOCATION under "eda_support_files/CONSTANTS.py"

In [1]:
%pip install osmnx contextily folium ipyleaflet cbsodata ydata-sdk ortools scikit-learn plotly matplotlib ipyleaflet
%pip install --upgrade "typing_extensions"

  Using cached contextily-1.7.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached ipyleaflet-0.20.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached cbsodata-1.3.5-py3-none-any.whl.metadata (7.8 kB)
  Using cached ydata_sdk-3.2.3-cp313-cp313-macosx_10_13_universal2.whl.metadata (40 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mercantile-1.2.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached pillow-12.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached rasterio-1.5.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (8.6 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Us

In [2]:
%load_ext autoreload
%autoreload 2

In [25]:
gemeente_to_run_for = "Amsterdam"

In [2]:
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional, Union

# def setup_logging(
#     level: int = logging.INFO,
#     log_file: Optional[Union[str, Path]] = None,
#     max_bytes: int = 10_000_000,  # 10 MB per file
#     backup_count: int = 5,         # keep last 5 rotated files
# ) -> None:
#     """
#     Configure application-wide logging for use inside the notebook.
#     Logs to console by default, and optionally to a rotating file.

#     Parameters
#     ----------
#     level : int
#         Logging level, e.g. logging.INFO or logging.DEBUG.
#     log_file : str | Path | None
#         Path to a log file. If provided, logs will be written to this file
#         in addition to the console. Parent directories are created if needed.
#     max_bytes : int
#         Maximum size per log file before rotation (bytes).
#     backup_count : int
#         Number of rotated log files to keep.
#     """
#     root_logger = logging.getLogger()
#     root_logger.setLevel(level)

#     # Common formatter
#     formatter = logging.Formatter(
#         "%(asctime)s %(levelname)s %(name)s - %(message)s",
#         datefmt="%Y-%m-%d %H:%M:%S"
#     )

#     # Prevent duplicate handlers on repeated cell execution
#     if not any(isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
#                for h in root_logger.handlers):
#         console_handler = logging.StreamHandler()
#         console_handler.setFormatter(formatter)
#         root_logger.addHandler(console_handler)

#     if log_file is not None:
#         log_path = Path(log_file)
#         log_path.parent.mkdir(parents=True, exist_ok=True)

#         # Only add file handler once per exact path
#         existing_file_handlers = [
#             h for h in root_logger.handlers
#             if isinstance(h, logging.FileHandler) and getattr(h, 'baseFilename', None) == str(log_path.resolve())
#         ]
#         if not existing_file_handlers:
#             file_handler = RotatingFileHandler(
#                 filename=str(log_path),
#                 maxBytes=max_bytes,
#                 backupCount=backup_count,
#                 encoding="utf-8"
#             )
#             file_handler.setFormatter(formatter)
#             root_logger.addHandler(file_handler)

# setup_logging(
#     level=logging.INFO,
#     log_file=r"/Workspace/Shared/nooddrinkwater_locaties_distributie/nooddrinkwater_locaties_distributie/logs/osm_parkeerplaatsen_EDA.log"
# )
# logger = logging.getLogger(__name__)

# # preventing py4j info logging by setting level higher
# logging.getLogger("py4j").setLevel(logging.WARNING)
# logging.getLogger("pyspark").setLevel(logging.WARNING)


In [3]:
from IPython.display import display, HTML
from eda_support_files.CONSTANTS import DATA_EXTERNAL_FOLDER_LOCATION, DATA_INTERIM_FOLDER_LOCATION, DATA_PROCESSED_FOLDER_LOCATION, DATA_RAW_FOLDER_LOCATION

# Add all necessary variables
ASSIGNED_RESIDENT_FOLDER_LOCATION = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents_assigned"
STATLINE_DIR = Path(f"{DATA_EXTERNAL_FOLDER_LOCATION}/statline_85618NED")
RUN_VISUALISATIONS = False

logging.getLogger("py4j").disabled = True

### Collecting and filtering the OpenStreetMap dataset for parking lots

#### For the case of 'nooddrinkwater' the eligible locations are parking lots, and OpenStreetMap is used to find these parking lots

In [4]:
from __future__ import annotations

import logging
from typing import Optional

import numpy as np
import geopandas as gpd

from eda_support_files.GetOSMData import GetOSMData


logger = logging.getLogger(__name__)
logger.info("Starting to load OpenStreetMap data")

# --- Load OSM data -----------------------------------------------------------

osm_data_getter = GetOSMData()
gdf_all: gpd.GeoDataFrame = osm_data_getter.run(
    folder=DATA_EXTERNAL_FOLDER_LOCATION,
    only_load=True
)

# Area of each geometry in square meters
if "area_m2" not in gdf_all.columns:
    gdf_all["area_m2"] = gdf_all.geometry.area.round(4)

# Perimeter / boundary length in meters
if "perimeter_m" not in gdf_all.columns:
    gdf_all["perimeter_m"] = gdf_all.geometry.length

# Polsby–Popper compactness measure "how square a parking lot is", to prevent long and small lots.
# (4πA) / P² where A = area, P = perimeter
if "compactness" not in gdf_all.columns:
    gdf_all["compactness"] = (
        4 * np.pi * gdf_all["area_m2"]
    ) / (gdf_all["perimeter_m"] ** 2)


#### Filtering the OSM parking lots to certain types (e.g. minimal size of parking lot)

In [5]:
from eda_support_files.FilterOSMData import FilterOSMData

osm_data_filterer = FilterOSMData(
        allowed_parking_types=None,
        min_parking_size=750.0,
        expected_crs_meters="EPSG:28992"
)
gdf_parking_lots = osm_data_filterer.run(gdf_all, folder=DATA_INTERIM_FOLDER_LOCATION, only_load=False)

# Fix crs for everything else after calculating area
gdf_parking_lots = gdf_parking_lots.to_crs(epsg=4326)

### Collecting and processing Gemeenten/wijken/buurten data (incl. geometry) via PDOK

#### Gemeente/wijk/buurten data is necessary for:
- the municipal boundaries
- data on number of residents (which determines the number of nooddrinkwaterpunten)  

In [6]:
from eda_support_files.GetGemeenteDataPDOK import GetGemeenteDataPDOK

gemeente_data_getter = GetGemeenteDataPDOK(filepath=DATA_RAW_FOLDER_LOCATION)
gdf_gemeenten_buurten = gemeente_data_getter.run(load_from_file=True)

display(gdf_gemeenten_buurten.sample(5))

,geometry,jrstatcode,jaar,buurtcode,buurtnaam,gemeentecode,gemeentenaam,aantal_inwoners,level
9162,"MULTIPOLYGON (((5.2735 51.65641, 5.2737 51.656...",2023BU08650004,2023.0,BU08650004,Villapark,GM0865,Vught,700,buurt
14592,"POLYGON ((4.64871 52.02057, 4.64863 52.02057, ...",NaN,NaN,NaN,NaN,GM0627,Waddinxveen,32620,gemeente
6690,"MULTIPOLYGON (((4.50705 52.16139, 4.50812 52.1...",2023BU05460109,2023.0,BU05460109,De Waard,GM0546,Leiden,2215,buurt
8805,"MULTIPOLYGON (((5.38681 51.68319, 5.38892 51.6...",2023BU08450308,2023.0,BU08450308,Verspreide huizen Beekveld-Hersend,GM0845,Sint-Michielsgestel,220,buurt
12087,"MULTIPOLYGON (((5.87433 51.34648, 5.87446 51.3...",2023BU18940508,2023.0,BU18940508,Verspreide huizen Steenoven en Langstraat,GM1894,Peel en Maas,305,buurt


In [7]:
import pandas as pd
from pathlib import Path

# --- CONFIG ---
left_df = gdf_gemeenten_buurten.copy()

# --- 1) Load Observations ---
obs = pd.read_csv(STATLINE_DIR / "Observations.csv",
                  sep=";", encoding="utf-8-sig", dtype=str)
obs = obs[["WijkenEnBuurten", "Measure", "Value"]].copy()
obs["WijkenEnBuurten"] = obs["WijkenEnBuurten"].str.strip()
obs["Measure"] = obs["Measure"].str.strip()

# --- 1.5) Value's can have a ',' which is a problem for pandas, so lets replace them with '.'
obs["Value"] = obs["Value"].str.replace(",", ".")
obs["Value"] = pd.to_numeric(obs["Value"], errors="coerce")

# Drop duplicates (region + measure)
obs = obs.drop_duplicates(subset=["WijkenEnBuurten", "Measure"])

# --- 2) Pivot to wide ---
wide = (
    obs.pivot_table(
        index="WijkenEnBuurten",
        columns="Measure",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

# --- 3) Load MeasureCodes and MeasureGroups ---
measures = pd.read_csv(STATLINE_DIR / "MeasureCodes.csv",
                       sep=";", encoding="utf-8-sig", dtype=str)[["Identifier", "Title", "MeasureGroupId"]]
groups = pd.read_csv(STATLINE_DIR / "MeasureGroups.csv",
                     sep=";", encoding="utf-8-sig", dtype=str)[["Id", "Title", "ParentId"]]
groups = groups.rename(columns={"Id": "MeasureGroupId", "Title": "GroupTitle"})

# Build lookups
group_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))
parent_lookup = dict(zip(groups["MeasureGroupId"], groups["ParentId"]))
parent_title_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))

# --- 4) Build combined label: Parent - Group - Measure ---
def build_full_label(row):
    group_id = row["MeasureGroupId"]
    group_title = group_lookup.get(group_id, "")
    parent_id = parent_lookup.get(group_id)
    parent_title = parent_title_lookup.get(parent_id, "") if pd.notna(parent_id) else ""
    parts = [p for p in [parent_title, group_title, row["Title"]] if p]
    return " - ".join(parts)

label_map = {row["Identifier"]: build_full_label(row) for _, row in measures.iterrows()}

# Rename columns in wide DataFrame
wide = wide.rename(columns=label_map)


# --- 5) Merge with your left_df ---
gdf_gemeenten_buurten_dem = left_df.merge(wide, left_on="buurtcode", right_on="WijkenEnBuurten", how="left")
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem.drop(columns=["WijkenEnBuurten"])
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem[[
    "geometry", "buurtnaam", "buurtcode", "gemeentenaam", "aantal_inwoners", "level"
]]

display(gdf_gemeenten_buurten_dem.sample(5))
gdf_gemeenten_buurten_dem.to_csv(f"{DATA_PROCESSED_FOLDER_LOCATION}/combined_dataset.csv", index=False)

,geometry,buurtnaam,buurtcode,gemeentenaam,aantal_inwoners,level
2939,"MULTIPOLYGON (((5.80284 51.81496, 5.80291 51.8...",Meijhorst,BU02680733,Nijmegen,3395,buurt
5498,"MULTIPOLYGON (((5.1942 52.75518, 5.19611 52.75...",Andijk Grootslag en IJsselhof,BU04201606,Medemblik,30,buurt
12344,"MULTIPOLYGON (((4.75218 52.08029, 4.75739 52.0...",Broekvelden-Noord,BU19010212,Bodegraven-Reeuwijk,1885,buurt
5379,"MULTIPOLYGON (((5.10759 52.65972, 5.10759 52.6...",Kersenboogerd-Zuid - Buurt 33 07,BU04053307,Hoorn,1945,buurt
2514,"MULTIPOLYGON (((6.29089 51.98132, 6.29108 51.9...",De Happert - Leerinkstraat,BU02220904,Doetinchem,1110,buurt


### Filter on Watergraafsmeer

In [8]:
watergraafsmeer = [
    "Drieburg",
    "Nieuwe Oosterbegraafplaats",
    "Betondorp",
    "Park de Meer",
    "Sportpark Middenmeer-Zuid",
    "Science Park-Zuid",
    "Sportpark Middenmeer-Noord",
    "De Wetbuurt",
    "Tuindorp Frankendael",
    "Middenmeer-Zuid",
    "Science Park-Noord",
    "Middenmeer-Noord",
    "Linnaeusparkbuurt",
    "Frankendael",
    "Don Bosco",
    "De Eenhoorn",
    "Julianapark",
    "Tuindorp Amstelstation",
    "Sportpark Voorland"
]


gdf_watergraafsmeer = gdf_gemeenten_buurten_dem[
    gdf_gemeenten_buurten_dem["buurtnaam"].isin(watergraafsmeer) & 
    (gdf_gemeenten_buurten_dem["gemeentenaam"] == "Amsterdam")
]
gdf_watergraafsmeer
gdf_watergraafsmeer.to_csv(f"{DATA_PROCESSED_FOLDER_LOCATION}/watergraafsmeer.csv", index=False)

### Calculate the necessary amounts of drinkwaterpunten for Watergraafsmeer

In [14]:
from eda_support_files.CalcBenodigdWaterpunt import CalcBenodigdWaterpunt

calc_benodigd_waterpunt_procesessor = CalcBenodigdWaterpunt(max_citizens_per_point = 2500)
gdf_watergraafsmeer = calc_benodigd_waterpunt_procesessor.run(gdf_watergraafsmeer)


### Select parking lots within Watergraafsmeer, sum how many in each gemeente and whether that is enough.

In [26]:
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData

gemeente_parking_combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_watergraafsmeer = gemeente_parking_combiner.run(
    gdf_parking_lots=gdf_parking_lots, 
    gdf_gemeenten=gdf_watergraafsmeer)

display(gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente_to_run_for].sample(5))

,access,parking,amenity,capacity,name,surface,operator,geometry,area_m2,perimeter_m,compactness,gemeentenaam
9459,customers,surface,parking,NaN,NaN,NaN,Intratuin,"POLYGON ((4.92562 52.34951, 4.92557 52.34954, ...",3927.9695,283.856248,0.612606,Amsterdam
49688,permit,surface,parking,NaN,NaN,NaN,NaN,"POLYGON ((4.9444 52.34982, 4.94431 52.34977, 4...",929.6866,158.214786,0.466716,Amsterdam
8006,NaN,surface,parking,282,NaN,compacted,Gemeente Amsterdam,"POLYGON ((4.93324 52.33964, 4.93323 52.33964, ...",4352.9393,1295.444341,0.032595,Amsterdam
33533,private,NaN,parking,55,NaN,NaN,NaN,"POLYGON ((4.95635 52.35682, 4.95641 52.35683, ...",1457.4907,223.454697,0.366806,Amsterdam
18143,private,surface,parking,91,NaN,NaN,NaN,"POLYGON ((4.9486 52.35469, 4.94863 52.35472, 4...",2515.9121,360.826941,0.242833,Amsterdam


In [16]:
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData


gemeente_parking_combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_gemeenten = gemeente_parking_combiner.run(
    gdf_parking_lots=gdf_parking_lots, 
    gdf_gemeenten=gdf_watergraafsmeer)

display(gdf_parking_lots.sample(5))

,access,parking,amenity,capacity,name,surface,operator,geometry,area_m2,perimeter_m,compactness,gemeentenaam
52206,yes,surface,parking,38,NaN,NaN,NaN,"POLYGON ((4.94346 52.3495, 4.94342 52.34952, 4...",871.7026,154.176400,0.460832,Amsterdam
16543,private,surface,parking,90,NaN,paving_stones,NaN,"POLYGON ((4.9596 52.35608, 4.95959 52.35612, 4...",2696.1437,587.755492,0.098075,Amsterdam
29076,private,surface,parking,75,NaN,NaN,APCOA Parking,"POLYGON ((4.95161 52.35527, 4.95165 52.35524, ...",1683.6819,220.660924,0.434529,Amsterdam
15740,private,surface,parking,102,NaN,NaN,APCOA Parking,"POLYGON ((4.94966 52.35657, 4.9499 52.35656, 4...",2793.7757,325.381120,0.331602,Amsterdam
5923,yes,surface,parking,NaN,NaN,NaN,NaN,"POLYGON ((4.95571 52.34538, 4.95588 52.34548, ...",5177.6314,441.261392,0.334156,Amsterdam


### Let's add the residents
To be able to calculate distances for residents to the nooddrinkwaterpunt, we need information on where these residents live.
We do not have this data available, but we use the BAG to find out which buildings (verblijfsobjecten) are for residential use (woonfunctie), and divide the residents over the residential verblijfsobjecten 

In [18]:
from eda_support_files.GenerateGemeenteResidents import GenerateGemeenteResidents

residents_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/watergraafsmeer.csv"

gemeente_residents_generator = GenerateGemeenteResidents(residents_folder = residents_folder)
# the generator creates and saves a file with resident locations
result_text = gemeente_residents_generator.run(
    gdf_gemeenten=gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == "Amsterdam"][['gemeentenaam']], 
    gdf_buurten=gdf_watergraafsmeer, 
    overwrite=True
)

### For each gemeente decide parking lots and resident assignment to parking lots

Experimenten:
- MinAvgDistanceSelector: minimize average distance from residents to parking lot
- MinMaxDistanceSelector: minimize maximum distance from residents to parking lot

Assignment:
The above described experiments only determine which parking lots to choose, hich resident has to go to which parking lot is not yet determined and is now determined with 'min_cost_flow', SimpleMinCostFlow from ORtools.

In [20]:
from eda_support_files.modelling.Orchestrator import ExperimentOrchestrator
import os

input_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents"

# Run for gemeenten in list
available_gemeente_file_paths = [f"{input_folder}/{gemeentennaam.replace(' ', '_')}.geojson" for gemeentennaam in list(["Amsterdam"])]

SETUP_EXPERIMENTS = {
    "minimum_avg_distance_min_cost_flow": {"optimisation_class": "MinAvgDistanceSelector", "assignment_method": "min_cost_flow"},
    "minimum_max_distance_min_cost_flow": {"optimisation_class": "MinMaxDistanceSelector", "assignment_method": "min_cost_flow"},
}

# Instantiate and run orchestrator
orchestrator = ExperimentOrchestrator(
    gemeente_filepaths = available_gemeente_file_paths,
    setup_experiments = SETUP_EXPERIMENTS,
    output_folder = ASSIGNED_RESIDENT_FOLDER_LOCATION,
    gdf_parking_lots = gdf_parking_lots,
    gdf_gemeenten = gdf_watergraafsmeer
)
orchestrator.run()


[Amsterdam] Starting (2 experiments)
[Amsterdam] → Skipping existing result: minimum_avg_distance_min_cost_flow
[Amsterdam] → Skipping existing result: minimum_max_distance_min_cost_flow
[Amsterdam] Finished: Completed

All municipalities processed.


### Use the code below to get info about experiments

In [22]:
import os
import geopandas as gpd
import pandas as pd

# Inputs
experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "gemeente_residents_assigned", "Amsterdam")

# Discover all .geojson files and load them
def load_experiments(folder: str, experiment_method: str = None) -> dict[str, gpd.GeoDataFrame]:
    """
    Loads all GeoJSON experiment files from the given folder.
    Returns a dict: {experiment_name: GeoDataFrame}
    """
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Experiment folder does not exist: {folder}")

    experiments = {}
    for fname in os.listdir(folder):
        # accept .geojson or .json, case-insensitive
        if fname.lower().endswith((".geojson", ".json")):
            experiment_name = os.path.splitext(fname)[0]  # strip extension
            # Skip if not experiment_method when given
            if experiment_method:
                if experiment_name != experiment_method:
                    continue
            fpath = os.path.join(folder, fname)
            try:
                gdf = gpd.read_file(fpath)
                experiments[experiment_name] = gdf
            except Exception as e:
                # Log and continue loading others
                logger.warning(f"[WARN] Failed to read '{fpath}': {e}")

    if not experiments:
        logger.info(f"[INFO] No experiment files found in: {folder}")

    return experiments

# Load all experiments
experiments = load_experiments(experiment_folder)

In [23]:
gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == "Amsterdam"]
gdf_gemeenten_gem = gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == "Amsterdam"]

display(HTML(f"<h4>{"Amsterdam"} has {len(experiments.get("minimum_avg_distance_min_cost_flow"))} residents</h2>"))
display(HTML(f"<h4>{"Amsterdam"} has {gdf_gemeenten_gem["aantal_parkeerplaatsen"].iloc[0]} eligible parking lots for nooddrinkwaterpunten and needs {gdf_gemeenten_gem["Benodigd_ceiling"].iloc[0]}</h2>"))

### Summary of evaluation metrics

In [24]:
from eda_support_files.summarize_results import summarize_assignments, visualize_distances, get_np_bins
from IPython.display import display, HTML
import plotly.graph_objects as go
import pandas as pd

# --- Build global bins across ALL experiments so histograms align
all_distances = pd.concat(
    [gdf["distance_to_parking"] for gdf in experiments.values()],
    ignore_index=True
)

# If get_np_bins expects two arrays, we can pass the same twice to derive bins from the whole set.
# Otherwise, if it accepts an iterable, change accordingly.
np_bins = get_np_bins(all_distances, all_distances)

# --- Create figure and plot each experiment
fig = go.Figure()

all_max_y = []
for exp_name, gdf in sorted(experiments.items()):
    display(HTML(f"<h3>Experiment: {exp_name}</h3>"))
    # Summary (table/metrics)
    summarize_assignments(
        gdf_res=gdf, 
        distances=gdf["distance_to_parking"])

    # Add histogram/trace to figure
    fig, max_y = visualize_distances(
        fig=fig,
        distances=gdf["distance_to_parking"],
        name=exp_name,
        np_bins=np_bins,
        color=None
    )
    all_max_y.append(max_y)
# Add the line for loopafstand
fig.add_shape(type="line", x0=1000, x1=1000, y0=0, y1=max(all_max_y), opacity=1,
                line=dict(color="black", width=4))
# --- Final layout for the combined plot
fig.update_layout(
    title="Distribution of resident-to-parking-lot distances (all experiments)",
    xaxis_title="Distance (m)",
    showlegend=True,
    bargap=0.1
)

fig.show()


## Export to GPKG

In [27]:
from eda_support_files.create_visualisation import create_interactive_map
import matplotlib.colors as mcolors
from matplotlib import colormaps
import numpy as np
import random

experiment_method_to_use = "minimum_avg_distance_min_cost_flow"

residents_nh = []
parking_lots_nh = []

for gemeente in gemeente_to_run_for:
    experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "gemeente_residents_assigned", gemeente)
    experiments = load_experiments(experiment_folder, experiment_method=experiment_method_to_use)

    # === Get residents data ===
    gdf_res_plot = experiments.get(experiment_method_to_use).copy()

    # === Prepare color mapping ===
    unique_lots = gdf_res_plot['assigned_parking_lot'].dropna().unique()
    random.shuffle(unique_lots)

    # Get colormap and sample colors
    cmap = colormaps.get_cmap('gist_rainbow')
    colors = [mcolors.to_hex(cmap(x)) for x in np.linspace(0, 1, len(unique_lots))]

    # Build mapping: lot ID → color
    lot_to_color = {int(lot): color for lot, color in zip(unique_lots, colors)}

    # Add color column to residents
    gdf_res_plot['color'] = gdf_res_plot['assigned_parking_lot'].map(lot_to_color)

    # === Prepare parking lots ===
    gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente]
    gdf_parking_lots_gem_selected = gdf_parking_lots_gem[
        gdf_parking_lots_gem.index.isin(unique_lots)
    ][['geometry']].copy()
    gdf_parking_lots_gem_selected['color'] = gdf_parking_lots_gem_selected.index.map(lot_to_color)

    ## Build GeoJSON features for parking lots
    parking_features = []
    for idx, row in gdf_parking_lots_gem_selected.iterrows():
        props = {"color": row["color"], "type": "parking_lot"}
        parking_features.append({
            "type": "Feature",
            "geometry": row.geometry.__geo_interface__,
            "properties": props
        })

    # === Config ===
    out_dir = f"{DATA_PROCESSED_FOLDER_LOCATION}/qgis_output/{experiment_method_to_use}/"
    gpkg_path = os.path.join(out_dir, "residents_parking_interview.gpkg")

    os.makedirs(out_dir, exist_ok=True)

    # === Copies of original data ===
    residents_copy = gdf_res_plot.copy()
    parking_copy = gdf_parking_lots_gem_selected.copy()

    # Ensure WGS84 CRS
    def to_wgs84(gdf):
        return (gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326))

    residents_copy = to_wgs84(residents_copy)
    parking_copy = to_wgs84(parking_copy)
    parking_copy = parking_copy.reset_index().rename(columns={"index": "parking_lot_id"})

    # Prepare columns for export
    residents_copy = residents_copy[["assigned_parking_lot", "color", "geometry"]]
    parking_copy = parking_copy[["parking_lot_id", "color", "geometry"]]
    # combine
    residents_nh.append(residents_copy)
    parking_lots_nh.append(parking_copy)

# combine all in list
residents_nh_comb = pd.concat(residents_nh).reset_index(drop=True)
parking_lots_nh_comb = pd.concat(parking_lots_nh).reset_index(drop=True)

# === Export GeoPackage ===
residents_nh_comb.to_file(gpkg_path, layer="residents", driver="GPKG")
parking_lots_nh_comb.to_file(gpkg_path, layer="parking_lots", driver="GPKG")

logger.info("✅ Export complete:")
logger.info(f"GeoPackage: {gpkg_path} (layers: residents, parking_lots)")


FileNotFoundError: Experiment folder does not exist: /Users/mayla/Desktop/hackaton_nipv_ams/data/interim/gemeente_residents_assigned/A